In [1]:
import os
import sys

sys.path.append('/home/cdsw/Tony/Mlops_new/Module')
import pickle

import config
import pandas as pd
from Model import get_model_feature_and_imp, get_model_path
from Sql_module import drop_SQL_raw_data, execute_sql, get_SQL_raw_data, send_table_to_sql

feature_trans_path = config.feature_trans_path


Bad key backend.qt4 in file /etc/matplotlib/matplotlibrc, line 43 ('backend.qt4 : PyQt4        # PyQt4 | PySide')
You probably need to get an updated matplotlibrc file from
https://github.com/matplotlib/matplotlib/blob/v3.3.4/matplotlibrc.template
or from the matplotlib source distribution


In [2]:
query = \
'''
select A.*,B.target,B.演算法,C.snap_date
from s_ianleong.mlops_retrain_log_double  A
left join s_ianleong.mlops_model_log_double B
on A.模型名稱 = B.模型名稱 and A.母體 = B.母體 and
ABS(TO_DATE(A."retrain日期",'YYYYMMDD HH24:MI') - TO_DATE(B."retrain日期",'YYYYMMDD HH24:MI')) <= 0.0007 and  A.版本 = B.版本
left join s_ianleong.mlops_ref_info_double C
on A.模型名稱 = concat(C.product,'模型') and A.母體 = C.population and B.target = C.target and  A.版本 = C.edition

'''

In [3]:
feat_num = 50
writine_table_name = 'mlops_feature_impt_double'

In [4]:
drop_SQL_raw_data(account=config.iaccount, pwd=config.ipwd, table_name=writine_table_name)

/home/cdsw/.local/lib/python3.6/site-packages/sqlalchemy/dialects/oracle/base.py:1412: SAWarning: Oracle version (19, 0, 0, 0, 0) is known to have a maximum identifier length of 128, rather than the historical default of 30. SQLAlchemy 1.4 will use 128 for this database; please set max_identifier_length=128 in create_engine() in order to test the application with this new length, or set to 30 in order to assure that 30 continues to be used.  In particular, pay close attention to the behavior of database migrations as dynamically generated names may change. See the section 'Max Identifier Lengths' in the SQLAlchemy Oracle dialect documentation for background.
  % ((self.server_version_info,))


mlops_feature_impt_double did not exist


In [5]:
model = get_SQL_raw_data(query, account=config.iaccount, pwd=config.ipwd)

Running time of : 0 sec
 loading completed


/home/cdsw/.local/lib/python3.6/site-packages/sqlalchemy/dialects/oracle/base.py:1412: SAWarning: Oracle version (19, 0, 0, 0, 0) is known to have a maximum identifier length of 128, rather than the historical default of 30. SQLAlchemy 1.4 will use 128 for this database; please set max_identifier_length=128 in create_engine() in order to test the application with this new length, or set to 30 in order to assure that 30 continues to be used.  In particular, pay close attention to the behavior of database migrations as dynamically generated names may change. See the section 'Max Identifier Lengths' in the SQLAlchemy Oracle dialect documentation for background.
  % ((self.server_version_info,))


In [6]:
def del_dupuli_end_word(r):
    if r['feature'][-2:] == '_x' or r['feature'][-2:] == '_y'or r['feature'][-2:] == '.1':
        return r['feature'][:-2]
    else:
        return r['feature']

In [7]:
feature_trans = get_SQL_raw_data(''' select distinct lower(特徵) as feature,business_glossory as 翻譯 from s_ianleong.new_feature_spec_2025
                                    union
                                    select distinct 特徵 as feature,翻譯 from s_ianleong.new_fin_feature_spec_2025''')
# feature_trans = feature_trans.rename(columns={'特徵':'feature'})

Running time of : 0 sec
 loading completed


/home/cdsw/.local/lib/python3.6/site-packages/sqlalchemy/dialects/oracle/base.py:1412: SAWarning: Oracle version (19, 0, 0, 0, 0) is known to have a maximum identifier length of 128, rather than the historical default of 30. SQLAlchemy 1.4 will use 128 for this database; please set max_identifier_length=128 in create_engine() in order to test the application with this new length, or set to 30 in order to assure that 30 continues to be used.  In particular, pay close attention to the behavior of database migrations as dynamically generated names may change. See the section 'Max Identifier Lengths' in the SQLAlchemy Oracle dialect documentation for background.
  % ((self.server_version_info,))


In [8]:
feature_trans

,feature,翻譯
0,age,年齡
1,agent_flag,交易代理人註記
2,all_login_close_m,任一電子平台最近一次登入距今月數
3,all_login_prod_12m,所有電子平台年登入次數
4,all_login_prod_12m_mdiff,所有電子平台年登入次數增減
...,...,...
5191,wti_mean_9M_diff,原油(WTI原油期貨)近9個月的changediff
5192,wti_mean_9M_rank,原油(WTI原油期貨)近9個月的相對水位(排名)
5193,wti_mean_9M_ratio,原油(WTI原油期貨)近9個月的changeratio
5194,years_from_open,往來年數


In [9]:
df = pd.DataFrame()
for rn in range(len(model)):
    the_retrain_log = model.iloc[rn]
    product = the_retrain_log['模型名稱'][:-2]
    writing_path = os.path.join('/home/cdsw/Tony/Mlops_new/審核通過模型_雙證',product)
    population = the_retrain_log['母體']
    new_edition = the_retrain_log['版本']
    target = the_retrain_log['target']
    algorithm = the_retrain_log['演算法']
    vali_ym = the_retrain_log['test_period']
    snap_date = the_retrain_log['snap_date']
    the_model_path = get_model_path(writing_path, product, population, new_edition, target, algorithm)
    the_model = pickle.load(open(the_model_path, 'rb'))
    feat_imp = get_model_feature_and_imp(the_model)
    feat_imp['feature'] = feat_imp.apply(del_dupuli_end_word, axis=1)
    feat_imp['importance'] = feat_imp.apply(lambda r: round(r['importance'],3), axis=1)
    model_imp = pd.merge(feat_imp, feature_trans, how='left', on=['feature'])
    fea_imptance = list(model_imp['importance'][:feat_num])
    fea_name = list(model_imp['翻譯'][:feat_num])
    fea_eng_name = list(model_imp['feature'][:feat_num])
    the_model_dict={}
    the_model_dict['product'] = product
    the_model_dict['population'] = population
    the_model_dict['target'] = target
    the_model_dict['edition'] = new_edition
    the_model_dict['algorithm'] = algorithm
    the_model_dict['snap_date'] = snap_date
    the_model_dict['vali_ym'] = vali_ym
    for index,fn in enumerate(fea_name):
        the_model_dict[f'feat_name_top{index}'] = fn
    for index,fen in enumerate(fea_eng_name):
        the_model_dict[f'feat_eng_name_top{index}'] = fen
    for index,fi in enumerate(fea_imptance):
        the_model_dict[f'feat_imptance_top{index}'] = fi
    df = df.append(pd.DataFrame([the_model_dict]))

In [10]:
model_imp

,index,importance,feature,翻譯
0,2,0.052,kyc_risk_desc,KYC等級
1,6,0.042,stwa_buy_amt_6m,認股權證半年買入金額
2,4,0.038,prod500_years_from_open,衍生性商品帳戶開戶距今年數
3,3,0.033,usd_avgasset_6m,美元計價商品半年平均庫存
4,0,0.026,stwa_txn_amt_6m,認股權證半年交易金額
...,...,...,...,...
96,68,0.003,sip_txn_amt_m,定期定額月交易金額
97,67,0.003,fp_turnover_12m_rdiff,財管商品年周轉率增減比例
98,86,0.003,twd_net_txn_6m_rdiff,台幣計價商品半年淨流入金額增減比例
99,98,0.002,fs_aum_6m_mdiff,海外股票半年庫存增減


In [11]:
df['snap_date'].fillna('未Predict', inplace=True)

In [12]:
missing_values = df.isnull()
row_with_missing = df[missing_values.any(axis=1)]
for i,row in row_with_missing.iterrows():
    for col in row_with_missing.columns:
        if pd.isnull(row[col]):
            na_name = row[col.replace('name','eng_name')]
            print(f'{col} is na, name is {na_name}')

feat_name_top1 is na, name is st_io_trans_amt_3m
feat_name_top2 is na, name is st_io_trans_amt_12m
feat_name_top3 is na, name is br_trans_num_m
feat_name_top7 is na, name is aum_m
feat_name_top8 is na, name is st_io_trans_amt_6m
feat_name_top9 is na, name is 台股變化_6M
feat_name_top11 is na, name is br_trans_amt_6m
feat_name_top14 is na, name is st_fd_buy_amt_12m_rdiff
feat_name_top18 is na, name is gs_buy_amt_3m_rdiff
feat_name_top21 is na, name is br_buy_num_12m
feat_name_top23 is na, name is br_trans_amt_12m
feat_name_top24 is na, name is gs_avg_safety_stock_3m
feat_name_top25 is na, name is AUM變化_12M
feat_name_top26 is na, name is od_buy_num_12m_mdiff
feat_name_top30 is na, name is od_trans_amt_12m_rdiff
feat_name_top33 is na, name is gs_avg_safety_stock_12m
feat_name_top38 is na, name is ap_bnf_12m_mdiff
feat_name_top39 is na, name is usd_invbnf_12m
feat_name_top41 is na, name is 台股變化_L
feat_name_top43 is na, name is st_bnf_3m_rdiff
feat_name_top49 is na, name is gs_trans_num_12m_rdi

feat_name_top1 is na, name is fp_trans_num_6m
feat_name_top2 is na, name is fp_avg_safety_stock_6m
feat_name_top4 is na, name is fp_avg_safety_stock_12m
feat_name_top5 is na, name is fp_avg_safety_stock_m
feat_name_top8 is na, name is fp_safety_stock_m
feat_name_top12 is na, name is fp_safety_stock_6m
feat_name_top13 is na, name is fp_avg_safety_stock_3m
feat_name_top14 is na, name is fp_buy_num_12m_mdiff
feat_name_top18 is na, name is os_buy_num_6m_mdiff
feat_name_top19 is na, name is bc_txn_prod_12m_mdiff
feat_name_top26 is na, name is gs_safety_stock_6m
feat_name_top27 is na, name is wa_buy_num_12m_mdiff
feat_name_top30 is na, name is st_io_trans_amt_12m
feat_name_top32 is na, name is fp_trans_num_12m_rdiff
feat_name_top35 is na, name is fp_buy_num_12m_rdiff
feat_name_top36 is na, name is lifeins_txn_day_12m_mdiff
feat_name_top38 is na, name is usd_avg_safety_stock_12m
feat_name_top40 is na, name is bc_txn_prod_12m_rdiff
feat_name_top44 is na, name is lifeins_txn_prod_12m_mdiff
feat

feat_name_top0 is na, name is fp_trans_num_6m
feat_name_top4 is na, name is fp_avg_safety_stock_6m
feat_name_top5 is na, name is fp_avg_safety_stock_12m
feat_name_top6 is na, name is kycqa_q94
feat_name_top11 is na, name is fp_invbnf_3m
feat_name_top12 is na, name is li_contract_due_mind
feat_name_top14 is na, name is fp_invbnf_6m
feat_name_top19 is na, name is li_month_max_buy_count
feat_name_top20 is na, name is li_contract_due_amt_mind
feat_name_top21 is na, name is sp_avg_safety_stock_m
feat_name_top25 is na, name is fp_trans_amt_12m_mdiff
feat_name_top29 is na, name is fp_trans_num_12m_mdiff
feat_name_top32 is na, name is usd_safety_stock_6m
feat_name_top33 is na, name is bc_months_max_avg_asset
feat_name_top35 is na, name is st_mo_trans_amt_12m_mdiff
feat_name_top46 is na, name is ap_invbnf_roi_6m
feat_name_top1 is na, name is fp_avg_safety_stock_m
feat_name_top2 is na, name is fp_trans_amt_12m
feat_name_top6 is na, name is fp_avg_safety_stock_6m
feat_name_top9 is na, name is li_

feat_name_top1 is na, name is st_io_trans_amt_m
feat_name_top3 is na, name is fo_avg_safety_stock_12m
feat_name_top10 is na, name is od_buy_num_12m_mdiff
feat_name_top11 is na, name is fo_avg_safety_stock_6m
feat_name_top12 is na, name is st_buy_num_3m
feat_name_top13 is na, name is br_buy_num_3m_mdiff
feat_name_top19 is na, name is st_etf_buy_amt_6m_mdiff
feat_name_top22 is na, name is br_safety_stock_12m
feat_name_top23 is na, name is bf_buy_num_12m_mdiff
feat_name_top28 is na, name is op_trans_num_12m_mdiff
feat_name_top32 is na, name is st_fd_safety_stock_6m
feat_name_top33 is na, name is ap_avg_safety_stock_12m
feat_name_top43 is na, name is twd_invbnf_12m
feat_name_top44 is na, name is op_trans_num_12m_rdiff
feat_name_top45 is na, name is wa_sell_amt_3m_mdiff
feat_name_top1 is na, name is sp_months_max_avg_asset
feat_name_top3 is na, name is sp_month_max_net_amt
feat_name_top5 is na, name is si_avg_safety_stock_3m
feat_name_top6 is na, name is sp_avg_safety_stock_12m
feat_name_to

feat_name_top0 is na, name is st_io_trans_amt_12m
feat_name_top12 is na, name is od_trans_amt_12m_mdiff
feat_name_top14 is na, name is dgt_acct_fct_cnt
feat_name_top16 is na, name is safety_stock_12m
feat_name_top17 is na, name is twd_trans_amt_12m_rdiff
feat_name_top19 is na, name is gs_avg_safety_stock_3m
feat_name_top23 is na, name is bc_txn_amt_12m
feat_name_top26 is na, name is li_contract_due_amt_mind
feat_name_top27 is na, name is si_avg_safety_stock_m
feat_name_top32 is na, name is kycqa_q98
feat_name_top33 is na, name is wa_safety_stock_3m_mdiff
feat_name_top38 is na, name is gs_safety_stock_3m
feat_name_top49 is na, name is ap_avg_safety_stock_3m
feat_name_top0 is na, name is fp_trans_num_6m
feat_name_top2 is na, name is fp_avg_safety_stock_12m
feat_name_top3 is na, name is kycqa_q92
feat_name_top4 is na, name is fp_avg_safety_stock_3m
feat_name_top5 is na, name is kycqa_q94
feat_name_top7 is na, name is od_trans_amt_12m
feat_name_top12 is na, name is fp_avg_safety_stock_6m
f

feat_name_top1 is na, name is fp_trans_amt_6m
feat_name_top3 is na, name is fp_avg_safety_stock_m
feat_name_top5 is na, name is bf_trans_num_12m_rdiff
feat_name_top10 is na, name is li_month_max_buy_count
feat_name_top11 is na, name is li_contract_due_amt_mind
feat_name_top12 is na, name is uu_months_from_txn
feat_name_top13 is na, name is fp_trans_amt_12m
feat_name_top18 is na, name is fo_avg_safety_stock_6m
feat_name_top23 is na, name is od_trans_amt_12m_rdiff
feat_name_top24 is na, name is kycqa_q8
feat_name_top25 is na, name is fp_avg_safety_stock_12m
feat_name_top29 is na, name is fu_sell_num_12m_mdiff
feat_name_top33 is na, name is gs_trans_num_3m_mdiff
feat_name_top34 is na, name is fp_buy_num_12m_rdiff
feat_name_top36 is na, name is kycqa_q97
feat_name_top39 is na, name is br_safety_stock_6m_mdiff
feat_name_top46 is na, name is fp_trans_num_12m_mdiff
feat_name_top15 is na, name is st_etf_buy_amt_m
feat_name_top18 is na, name is od_avg_safety_stock_12m
feat_name_top23 is na, nam

feat_name_top13 is na, name is dgt_stcqt_tp5_cnt
feat_name_top18 is na, name is dgt_waif_cnt
feat_name_top19 is na, name is od_trans_num_12m_mdiff
feat_name_top28 is na, name is op_trans_num_12m_mdiff
feat_name_top31 is na, name is safety_stock_12m
feat_name_top32 is na, name is fo_avg_safety_stock_m
feat_name_top38 is na, name is dgt_stcqt_tp5_cnt_m_mdiff
feat_name_top39 is na, name is od_trans_num_6m_mdiff
feat_name_top41 is na, name is st_etf_buy_num_12m_mdiff
feat_name_top44 is na, name is gs_trans_num_12m_mdiff
feat_name_top45 is na, name is dgt_afthif_cnt_ratio
feat_name_top3 is na, name is fp_safety_stock_m
feat_name_top5 is na, name is sp_avg_safety_stock_12m
feat_name_top11 is na, name is op_sell_num_12m
feat_name_top13 is na, name is sp_months_max_avg_asset
feat_name_top14 is na, name is ba_fd_safety_stock_3m
feat_name_top20 is na, name is usd_safety_stock_3m_rdiff
feat_name_top21 is na, name is usd_safety_stock_12m_mdiff
feat_name_top24 is na, name is kycqa_q95
feat_name_top

feat_name_top0 is na, name is fp_trans_amt_12m
feat_name_top2 is na, name is fp_safety_stock_m
feat_name_top3 is na, name is fp_avg_safety_stock_12m
feat_name_top4 is na, name is li_month_max_buy_count
feat_name_top5 is na, name is fp_trans_num_12m
feat_name_top7 is na, name is li_month_max_buy_amt
feat_name_top8 is na, name is ipo_trans_amt_12m
feat_name_top11 is na, name is st_mo_trans_num_12m
feat_name_top13 is na, name is li_contract_due_amt_mind
feat_name_top15 is na, name is od_trans_num_12m_rdiff
feat_name_top17 is na, name is fd_mo_trans_amt_12m
feat_name_top19 is na, name is in_wl_trans_amt_12m_mdiff
feat_name_top21 is na, name is st_mo_trans_amt_12m
feat_name_top24 is na, name is ap_avg_safety_stock_12m
feat_name_top26 is na, name is gs_avg_safety_stock_6m
feat_name_top28 is na, name is kycqa_q8
feat_name_top29 is na, name is st_etf_buy_amt_12m
feat_name_top31 is na, name is kycqa_q93
feat_name_top32 is na, name is gs_trans_num_12m_mdiff
feat_name_top35 is na, name is dgt_stc

feat_name_top9 is na, name is dgt_stcqt_tp5_cnt
feat_name_top12 is na, name is dgt_waif_cnt
feat_name_top17 is na, name is st_io_trans_num_3m
feat_name_top18 is na, name is dgt_tca_5line_cnt
feat_name_top19 is na, name is dgt_acct_fo_cnt_m_mdiff
feat_name_top21 is na, name is dgt_acct_fo_cnt
feat_name_top32 is na, name is os_buy_num_12m_mdiff
feat_name_top33 is na, name is dgt_stcqt_tp5_cnt_m_mdiff
feat_name_top36 is na, name is cf_safety_stock_6m_rdiff
feat_name_top37 is na, name is wa_trans_amt_12m_mdiff
feat_name_top38 is na, name is ap_avg_safety_stock_12m
feat_name_top40 is na, name is fs_io_trans_amt_6m
feat_name_top42 is na, name is fo_avg_safety_stock_3m
feat_name_top44 is na, name is wa_buy_amt_12m_mdiff
feat_name_top0 is na, name is st_etf_trans_amt_3m
feat_name_top5 is na, name is st_etf_avg_safety_stock_m
feat_name_top14 is na, name is gs_trans_amt_12m
feat_name_top17 is na, name is st_etf_avg_safety_stock_12m
feat_name_top19 is na, name is st_etf_buy_amt_3m
feat_name_top20

In [13]:
# m1 = df['product'] == '投資型保險商品'
# m2 = df['edition'].isin(['1.5','2.1'])

# df[m1 & m2]

In [13]:
df

,product,population,target,edition,algorithm,snap_date,vali_ym,feat_name_top0,feat_name_top1,feat_name_top2,...,feat_imptance_top40,feat_imptance_top41,feat_imptance_top42,feat_imptance_top43,feat_imptance_top44,feat_imptance_top45,feat_imptance_top46,feat_imptance_top47,feat_imptance_top48,feat_imptance_top49
0,不限用途,潛客,新戶開發與靜止戶活化,1.1,xgboost,未Predict,202305,所有商品季交易金額,NaN,NaN,...,0.009,0.009,0.009,0.008,0.008,0.008,0.008,0.008,0.008,0.008
0,保險商品,非潛客,新戶開發與靜止戶活化,1.1,xgboost,未Predict,202305,NaN,財管商品年淨流入金額增減,平衡多重基金季周轉率,...,0.009,0.009,0.009,0.009,0.009,0.008,0.008,0.008,0.008,0.008
0,儲蓄型保險商品,非潛客,新戶開發與靜止戶活化,1.1,xgboost,未Predict,202305,NaN,財管商品年淨流入金額增減,NaN,...,0.009,0.009,0.009,0.009,0.009,0.009,0.009,0.009,0.008,0.008
0,雙向借券,非潛客,新戶開發與靜止戶活化,1.1,xgboost,未Predict,202305,NaN,NaN,NaN,...,0.006,0.006,0.006,0.006,0.006,0.006,0.005,0.005,0.005,0.005
0,雙向借券,潛客,新戶開發與靜止戶活化,1.1,xgboost,未Predict,202305,NaN,NaN,NaN,...,0.004,0.004,0.004,0.004,0.004,0.003,0.003,0.003,0.003,0.003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,台股信用交易,潛客,新戶開發與靜止戶活化,4.18,xgboost,未Predict,202511,台股現貨半年周轉率,所有商品季交易金額,所有商品半年賣出次數,...,0.005,0.005,0.005,0.005,0.005,0.004,0.004,0.004,0.004,0.004
0,股票型基金,潛客,新戶開發與靜止戶活化,3.19,xgboost,未Predict,202511,信託帳戶有效戶註記,基金歷史最高月均庫存金額,基金歷史最高單月交易量,...,0.007,0.006,0.006,0.006,0.006,0.006,0.006,0.006,0.006,0.006
0,債券型基金,潛客,新戶開發與靜止戶活化,4.18,xgboost,未Predict,202511,信託帳戶有效戶註記,基金歷史最高月均庫存金額,KYC問題8_4,...,0.009,0.009,0.009,0.009,0.009,0.008,0.008,0.008,0.008,0.008
0,基金定期定額,潛客,新戶開發與靜止戶活化,1.17,xgboost,未Predict,202511,信託帳戶有效戶註記,基金歷史最高月均庫存金額,財管商品月平均庫存,...,0.007,0.007,0.007,0.006,0.006,0.006,0.006,0.006,0.006,0.006


In [14]:
group = df.groupby('snap_date').count()
group

,product,population,target,edition,algorithm,vali_ym,feat_name_top0,feat_name_top1,feat_name_top2,feat_name_top3,...,feat_imptance_top40,feat_imptance_top41,feat_imptance_top42,feat_imptance_top43,feat_imptance_top44,feat_imptance_top45,feat_imptance_top46,feat_imptance_top47,feat_imptance_top48,feat_imptance_top49
snap_date,,,,,,,,,,,,,,,,,,,,,
2023/08/31,32,32,32,32,32,32,16,18,19,17,...,32,32,32,32,32,32,32,32,32,32
2023/09/30,32,32,32,32,32,32,19,15,13,19,...,32,32,32,32,32,32,32,32,32,32
2023/10/31,32,32,32,32,32,32,12,21,13,18,...,32,32,32,32,32,32,32,32,32,32
2023/11/30,34,34,34,34,34,34,19,15,23,21,...,34,34,34,34,34,34,34,34,34,34
2023/12/31,34,34,34,34,34,34,18,19,21,21,...,34,34,34,34,34,34,34,34,34,34
2024/01/31,34,34,34,34,34,34,20,16,20,20,...,34,34,34,34,34,34,34,34,34,34
2024/02/29,34,34,34,34,34,34,21,17,21,21,...,34,34,34,34,34,34,34,34,34,34
2024/03/31,34,34,34,34,34,34,23,19,21,19,...,34,34,34,34,34,34,34,34,34,34
2024/04/30,34,34,34,34,34,34,22,19,21,24,...,34,34,34,34,34,34,34,34,34,34


In [15]:
writine_table_name

'mlops_feature_impt_double'

In [17]:
# write_data_to_SQL(table_name=writine_table_name, df=df, account=config.account_yichieh, pwd=config.pwd_yichieh, exist_action='replace', chunk_size=10000, col_types=col_types)

In [16]:
send_table_to_sql(targ=df,table_name='mlops_feature_impt_double',account=config.iaccount, pwd=config.ipwd)

/home/cdsw/.local/lib/python3.6/site-packages/sqlalchemy/dialects/oracle/base.py:1412: SAWarning: Oracle version (19, 0, 0, 0, 0) is known to have a maximum identifier length of 128, rather than the historical default of 30. SQLAlchemy 1.4 will use 128 for this database; please set max_identifier_length=128 in create_engine() in order to test the application with this new length, or set to 30 in order to assure that 30 continues to be used.  In particular, pay close attention to the behavior of database migrations as dynamically generated names may change. See the section 'Max Identifier Lengths' in the SQLAlchemy Oracle dialect documentation for background.
  % ((self.server_version_info,))


[Success writing down to db] Running time : 1 sec


/usr/local/lib/python3.6/site-packages/pandas/io/sql.py:1336: UserWarning: The provided table name 'MLOPS_FEATURE_IMPT_DOUBLE' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  warnings.warn(msg, UserWarning)


In [17]:
grant_query = \
f'''
grant select on {writine_table_name} to 
 s_florakao, s_benwen, s_andyctkuo, s_tommytung, s_alexwcchung,s_tlyu,
  s_paohsiangwang, s_dingwenchen, s_summerccchen,s_evelynnchou,s_tomhhuang,s_krisyjchen,s_ianleong
'''

In [18]:
print(grant_query)


grant select on mlops_feature_impt_double to 
 s_florakao, s_benwen, s_andyctkuo, s_tommytung, s_alexwcchung,s_tlyu,
  s_paohsiangwang, s_dingwenchen, s_summerccchen,s_evelynnchou,s_tomhhuang,s_krisyjchen,s_ianleong



In [19]:
execute_sql(query, account=config.account, pwd=config.pwd)

Running time of : 0 sec
 executing completed


/home/cdsw/.local/lib/python3.6/site-packages/sqlalchemy/dialects/oracle/base.py:1412: SAWarning: Oracle version (19, 0, 0, 0, 0) is known to have a maximum identifier length of 128, rather than the historical default of 30. SQLAlchemy 1.4 will use 128 for this database; please set max_identifier_length=128 in create_engine() in order to test the application with this new length, or set to 30 in order to assure that 30 continues to be used.  In particular, pay close attention to the behavior of database migrations as dynamically generated names may change. See the section 'Max Identifier Lengths' in the SQLAlchemy Oracle dialect documentation for background.
  % ((self.server_version_info,))


In [21]:
# grant_query_mlops_id_list_double_wold = \
# f'''
# grant select on mlops_ref_info_double_wold to
#  s_florakao, s_benwen, s_arthuryjhuang, s_minshengchen, s_andyctkuo, s_tommytung, s_alexwcchung,
#  s_skyhuang, s_renderwang, s_paohsiangwang, s_dingwenchen, s_jhihweichen, s_summerccchen, s_taichiehfan
# '''

In [22]:
# execute_sql(grant_query_mlops_id_list_double_wold )

In [20]:
execute_sql(grant_query, account=config.account, pwd=config.pwd)

Running time of : 0 sec
 executing completed


/home/cdsw/.local/lib/python3.6/site-packages/sqlalchemy/dialects/oracle/base.py:1412: SAWarning: Oracle version (19, 0, 0, 0, 0) is known to have a maximum identifier length of 128, rather than the historical default of 30. SQLAlchemy 1.4 will use 128 for this database; please set max_identifier_length=128 in create_engine() in order to test the application with this new length, or set to 30 in order to assure that 30 continues to be used.  In particular, pay close attention to the behavior of database migrations as dynamically generated names may change. See the section 'Max Identifier Lengths' in the SQLAlchemy Oracle dialect documentation for background.
  % ((self.server_version_info,))
